# document_detail

> Document detail dashboard with info, stats, integrity checks, and samples

In [ ]:
#| default_exp components.document_detail

In [ ]:
#| export
from typing import Any, List, Optional

from fasthtml.common import Div, Span, H2, H3, P, A, Button

# DaisyUI components
from cjm_fasthtml_daisyui.components.actions.button import (
    btn, btn_colors, btn_styles, btn_sizes
)
from cjm_fasthtml_daisyui.components.data_display.card import card, card_body
from cjm_fasthtml_daisyui.utilities.semantic_colors import (
    bg_dui, text_dui, shadow_dui, border_dui
)

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, min_w
from cjm_fasthtml_tailwind.utilities.typography import (
    font_size, font_weight, font_family, truncate
)
from cjm_fasthtml_tailwind.utilities.effects import shadow
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, flex_wrap, flex,
    items, justify, gap
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Local imports
from cjm_transcript_workflow_management.models import (
    DocumentDetail, SegmentSample, ManagementUrls
)
from cjm_transcript_workflow_management.html_ids import ManagementHtmlIds
from cjm_transcript_workflow_management.utils import (
    format_duration, format_duration_short, format_date,
    format_datetime, truncate_text, format_time_range
)
from cjm_transcript_workflow_management.components.helpers import (
    render_section_header, render_delete_modal,
    DEBUG_MANAGEMENT_RENDER
)

## Shared Helpers

In [ ]:
#| export
# Card styling constants
_CARD_CLS = combine_classes(card, bg_dui.base_200, shadow.sm, shadow_dui.primary)

def _render_stat_row(
    label:str,  # Label text
    value:str,  # Value text
) -> Any:  # Flexbox row element
    """Render a label-value row for stat display."""
    return Div(
        Span(label, cls=str(text_dui.base_content.opacity(70))),
        Span(value, cls=str(font_weight.medium)),
        cls=combine_classes(flex_display, justify.between)
    )

def _render_check_row(
    passed:bool,  # Whether the check passed
    label:str,  # Check description
    detail:str="",  # Optional detail text (e.g., counts)
) -> Any:  # Flexbox row with icon
    """Render a pass/fail check row with icon."""
    if passed:
        icon = lucide_icon("circle-check", size=4, cls=str(text_dui.success))
    else:
        icon = lucide_icon("triangle-alert", size=4, cls=str(text_dui.warning))
    
    children = [icon, Span(label, cls=str(font_size.sm))]
    if detail:
        children.append(Span(
            detail,
            cls=combine_classes(font_size.sm, text_dui.base_content.opacity(60))
        ))
    
    return Div(
        *children,
        cls=combine_classes(flex_display, items.center, gap(2), p.y(1))
    )

In [ ]:
from fasthtml.common import to_xml

row = _render_stat_row("Segments", "247")
xml = to_xml(row)
assert "Segments" in xml
assert "247" in xml

check = _render_check_row(True, "STARTS_WITH edge", "1")
xml = to_xml(check)
assert "circle-check" in xml or "svg" in xml
assert "STARTS_WITH" in xml
print("Helpers OK")

Helpers OK


## Detail Header

Back to List button, title, and action buttons.

In [ ]:
#| export
def render_detail_header(
    detail:DocumentDetail,  # Document detail data
    urls:ManagementUrls,  # URL bundle for route endpoints
) -> Any:  # Header element with navigation and actions
    """Render the detail view header with Back, Export, and Delete buttons."""
    return Div(
        # Left: Back to List
        Button(
            lucide_icon("arrow-left", size=4),
            "Back to List",
            cls=combine_classes(
                btn, btn_styles.ghost, btn_sizes.sm,
                flex_display, items.center, gap(1)
            ),
            hx_get=urls.management_page,
            hx_target=f"#{ManagementHtmlIds.PAGE_CONTENT}",
            hx_swap="outerHTML",
        ),
        # Right: Export + Delete
        Div(
            # Export: plain link for file download
            A(
                lucide_icon("download", size=4),
                "Export",
                cls=combine_classes(
                    btn, btn_styles.outline, btn_sizes.sm,
                    flex_display, items.center, gap(1)
                ),
                href=f"{urls.export_document}?doc_id={detail.document_id}",
                download=True,
            ),
            Button(
                lucide_icon("trash-2", size=4),
                "Delete",
                cls=combine_classes(
                    btn, btn_colors.error, btn_styles.outline, btn_sizes.sm,
                    flex_display, items.center, gap(1)
                ),
                onclick=(
                    f"prepareDetailDeleteModal('{detail.document_id}', '{detail.title}', {detail.segment_count}); "
                    f"document.getElementById('{ManagementHtmlIds.DELETE_MODAL}').showModal()"
                ),
            ),
            cls=combine_classes(flex_display, gap(2)),
        ),
        cls=combine_classes(flex_display, justify.between, items.center, m.b(4)),
    )

In [ ]:
from fasthtml.common import to_xml
from cjm_transcript_workflow_management.models import DocumentDetail, ManagementUrls

urls = ManagementUrls(
    management_page="/manage/documents/management_page",
    list_documents="/manage/documents/list_documents",
    document_detail="/manage/documents/document_detail",
    delete_document="/manage/documents/delete_document",
    delete_selected="/manage/documents/delete_selected",
    export_document="/manage/export/export_document",
    export_all="/manage/export/export_all",
    import_graph="/manage/import/import_graph",
)

detail = DocumentDetail(
    document_id="abc-123-def-456", title="1. Laying Plans",
    media_type="audio", created_at=1740000000.0, updated_at=1740000000.0,
    segment_count=247, total_duration=2537.0, avg_segment_duration=10.3,
    has_starts_with=True, next_chain_complete=True, next_count=246,
    part_of_complete=True, part_of_count=247,
    all_have_timing=True, segments_missing_timing=0,
    all_have_sources=True, segments_missing_sources=0,
    all_checks_passed=True,
    source_plugins=["cjm-transcription-plugin-whisper"],
    first_segments=[SegmentSample(0, "Sun Tzu said,", 0.0, 2.1)],
    last_segments=[SegmentSample(246, "Hence under these five...", 2528.0, 2537.0)],
)

hdr = render_detail_header(detail, urls)
xml = to_xml(hdr)
assert "Back to List" in xml
assert "Export" in xml
assert "Delete" in xml
assert urls.management_page in xml
assert 'hx-swap="outerHTML"' in xml
assert f'href="{urls.export_document}?doc_id={detail.document_id}"' in xml
assert "download" in xml
print("Detail header OK")

## Document Info Section

In [ ]:
#| export
def render_document_info(
    detail:DocumentDetail,  # Document detail data
) -> Any:  # Card element with document info
    """Render the document info card."""
    return Div(
        Div(
            render_section_header("Document", "file-text"),
            Div(
                _render_stat_row("Title", detail.title),
                _render_stat_row("Media Type", detail.media_type),
                _render_stat_row("ID", f"{detail.document_id[:12]}..."),
                _render_stat_row("Created", format_datetime(detail.created_at)),
                _render_stat_row("Updated", format_datetime(detail.updated_at)),
                cls=combine_classes(flex_display, flex_direction.col, gap(1), m.t(3)),
            ),
            cls=str(card_body)
        ),
        cls=combine_classes(_CARD_CLS, flex._1, min_w("280px")),
    )

In [ ]:
info = render_document_info(detail)
xml = to_xml(info)
assert "1. Laying Plans" in xml
assert "abc-123-def-" in xml
assert "audio" in xml
print("Document info OK")

Document info OK


## Segment Stats Section

In [ ]:
#| export
def render_segment_stats(
    detail:DocumentDetail,  # Document detail data
) -> Any:  # Card element with segment stats
    """Render the segment statistics card."""
    return Div(
        Div(
            render_section_header("Segments", "list"),
            Div(
                _render_stat_row("Count", str(detail.segment_count)),
                _render_stat_row("Total Duration", format_duration(detail.total_duration)),
                _render_stat_row("Average Duration", format_duration_short(detail.avg_segment_duration)),
                cls=combine_classes(flex_display, flex_direction.col, gap(1), m.t(3)),
            ),
            cls=str(card_body)
        ),
        cls=combine_classes(_CARD_CLS, flex._1, min_w("280px")),
    )

In [ ]:
stats = render_segment_stats(detail)
xml = to_xml(stats)
assert "247" in xml
assert "42:17" in xml
assert "10.3s" in xml
print("Segment stats OK")

Segment stats OK


## Source Traceability Section

In [ ]:
#| export
def render_sources_info(
    detail:DocumentDetail,  # Document detail data
) -> Any:  # Card element with source plugin info
    """Render the source traceability card."""
    plugins_text = ", ".join(detail.source_plugins) if detail.source_plugins else "None"
    
    return Div(
        Div(
            render_section_header("Sources", "database"),
            Div(
                _render_stat_row("Plugins", plugins_text),
                cls=combine_classes(flex_display, flex_direction.col, gap(1), m.t(3)),
            ),
            cls=str(card_body)
        ),
        cls=combine_classes(_CARD_CLS, flex._1, min_w("280px")),
    )

In [ ]:
src = render_sources_info(detail)
xml = to_xml(src)
assert "cjm-transcription-plugin-whisper" in xml
print("Sources info OK")

Sources info OK


## Integrity Checks Section

In [ ]:
#| export
def render_integrity_checks(
    detail:DocumentDetail,  # Document detail data
) -> Any:  # Card element with integrity check rows
    """Render the integrity checks card with pass/fail indicators."""
    expected_next = max(0, detail.segment_count - 1)
    
    return Div(
        Div(
            render_section_header("Structure Integrity", "shield-check"),
            Div(
                _render_check_row(
                    detail.has_starts_with,
                    "STARTS_WITH edge",
                    "1" if detail.has_starts_with else "0"
                ),
                _render_check_row(
                    detail.next_chain_complete,
                    "NEXT chain",
                    f"{detail.next_count}/{expected_next}"
                ),
                _render_check_row(
                    detail.part_of_complete,
                    "PART_OF edges",
                    f"{detail.part_of_count}/{detail.segment_count}"
                ),
                _render_check_row(
                    detail.all_have_timing,
                    "All segments have timing",
                    f"{detail.segments_missing_timing} missing" if not detail.all_have_timing else ""
                ),
                _render_check_row(
                    detail.all_have_sources,
                    "All segments have sources",
                    f"{detail.segments_missing_sources} missing" if not detail.all_have_sources else ""
                ),
                cls=combine_classes(flex_display, flex_direction.col, m.t(3)),
            ),
            cls=str(card_body)
        ),
        cls=str(_CARD_CLS),
    )

In [ ]:
checks = render_integrity_checks(detail)
xml = to_xml(checks)
assert "STARTS_WITH" in xml
assert "NEXT chain" in xml
assert "PART_OF" in xml
assert "246/246" in xml
assert "247/247" in xml
print("Integrity checks OK")

Integrity checks OK


## Sample Segments Section

In [ ]:
#| export
def _render_sample_row(
    sample:SegmentSample,  # Segment sample data
) -> Any:  # Flexbox row with index, text, and timing
    """Render a single sample segment row."""
    return Div(
        Span(
            f"[{sample.index}]",
            cls=combine_classes(font_family.mono, text_dui.primary, font_size.sm)
        ),
        Span(
            f'"{truncate_text(sample.text, 50)}"',
            cls=combine_classes(flex._1, font_size.sm)
        ),
        Span(
            f"({format_time_range(sample.start_time, sample.end_time)})",
            cls=combine_classes(
                font_family.mono, font_size.xs,
                text_dui.base_content.opacity(60)
            )
        ),
        cls=combine_classes(flex_display, items.center, gap(2), p.y(1)),
    )

def _render_sample_list(
    samples:List[SegmentSample],  # List of segment samples
    label:str,  # Section label (e.g., "First", "Last")
) -> Any:  # Flexbox column with label and rows
    """Render a labeled list of sample segments."""
    if not samples:
        return Div(
            Span(f"{label}:", cls=combine_classes(font_weight.medium, font_size.sm)),
            P("No samples", cls=str(text_dui.base_content.opacity(50))),
        )
    
    return Div(
        Span(f"{label}:", cls=combine_classes(font_weight.medium, font_size.sm, m.b(1))),
        *[_render_sample_row(s) for s in samples],
        cls=combine_classes(flex_display, flex_direction.col),
    )

In [ ]:
#| export
def render_sample_segments(
    detail:DocumentDetail,  # Document detail data
) -> Any:  # Card element with sample segment lists
    """Render the sample segments card with first and last segments."""
    return Div(
        Div(
            render_section_header("Sample Segments", "scan-search"),
            Div(
                _render_sample_list(detail.first_segments, "First"),
                Div(cls=str(m.y(3))),
                _render_sample_list(detail.last_segments, "Last"),
                cls=combine_classes(m.t(3)),
            ),
            cls=str(card_body)
        ),
        cls=str(_CARD_CLS),
    )

In [ ]:
samples = render_sample_segments(detail)
xml = to_xml(samples)
assert "[0]" in xml
assert "Sun Tzu said," in xml
assert "[246]" in xml
assert "First:" in xml
assert "Last:" in xml
print("Sample segments OK")

Sample segments OK


## Detail Scripts

JavaScript for delete from detail view.

In [ ]:
#| export
def render_detail_scripts(
    urls:ManagementUrls,  # URL bundle for route endpoints
) -> Any:  # Script element
    """Render client-side JavaScript for delete from detail view."""
    from fasthtml.common import Script
    return Script(f"""
    function prepareDetailDeleteModal(docId, title, segCount) {{
        var body = document.getElementById('{ManagementHtmlIds.DELETE_MODAL_BODY}');
        if (body) {{
            body.innerHTML = '<p>Permanently delete "' + title + '" and all ' + segCount + ' segments?</p>'
                + '<p class="{combine_classes(font_size.sm, text_dui.base_content.opacity(60), m.t(2))}">This action cannot be undone.</p>';
        }}
        var confirmBtn = document.querySelector('#{ManagementHtmlIds.DELETE_MODAL} [data-delete-confirm]');
        if (confirmBtn) {{
            confirmBtn.setAttribute('hx-post', '{urls.delete_document}');
            confirmBtn.setAttribute('hx-vals', JSON.stringify({{doc_id: docId, return_page: 'true'}}));
            confirmBtn.setAttribute('hx-target', '#{ManagementHtmlIds.PAGE_CONTENT}');
            confirmBtn.setAttribute('hx-swap', 'outerHTML');
            htmx.process(confirmBtn);
        }}
    }}
    """)

In [ ]:
scripts = render_detail_scripts(urls)
xml = to_xml(scripts)
assert "prepareDetailDeleteModal" in xml
print("Detail scripts OK")

Detail scripts OK


## Full Detail View

Complete detail dashboard composing all sections.

In [ ]:
#| export
def render_document_detail(
    detail:DocumentDetail,  # Document detail data
    urls:ManagementUrls,  # URL bundle for route endpoints
) -> Any:  # Complete detail dashboard
    """Render the complete document detail dashboard."""
    if DEBUG_MANAGEMENT_RENDER:
        print(f"[RENDER] document_detail: {detail.title}")
    
    return Div(
        # Header with navigation and actions
        render_detail_header(detail, urls),
        # Title
        H2(
            detail.title,
            cls=combine_classes(font_size._2xl, font_weight.bold, m.b(4))
        ),
        # Summary cards row
        Div(
            render_document_info(detail),
            render_segment_stats(detail),
            render_sources_info(detail),
            cls=combine_classes(flex_display, flex_wrap.wrap, gap(4), m.b(4)),
        ),
        # Integrity checks
        render_integrity_checks(detail),
        # Sample segments
        render_sample_segments(detail),
        # Delete modal
        render_delete_modal(
            modal_id=ManagementHtmlIds.DELETE_MODAL,
            body_id=ManagementHtmlIds.DELETE_MODAL_BODY,
            title="Delete Document?",
            confirm_attrs={
                "data_delete_confirm": "true",
                "onclick": f"document.getElementById('{ManagementHtmlIds.DELETE_MODAL}').close()",
            },
        ),
        # Scripts
        render_detail_scripts(urls),
        id=ManagementHtmlIds.DOCUMENT_DETAIL,
        cls=combine_classes(flex_display, flex_direction.col, gap(4)),
    )

In [ ]:
full_detail = render_document_detail(detail, urls)
xml = to_xml(full_detail)
assert ManagementHtmlIds.DOCUMENT_DETAIL in xml
assert "1. Laying Plans" in xml
assert "Back to List" in xml
assert "STARTS_WITH" in xml
assert "[0]" in xml
assert "Sun Tzu said," in xml
print(f"Full detail OK ({len(xml)} chars)")

Full detail OK (10863 chars)


## Error State

In [ ]:
#| export
def render_detail_error(
    message:str="Document not found.",  # Error message
    urls:ManagementUrls=None,  # URL bundle for Back to List
) -> Any:  # Error state element
    """Render an error state for the detail view."""
    children = [
        lucide_icon("circle-alert", size=12, cls=str(text_dui.error)),
        P(message, cls=combine_classes(font_size.lg, font_weight.medium, m.t(4))),
    ]
    if urls:
        children.append(Button(
            lucide_icon("arrow-left", size=4),
            "Back to List",
            cls=combine_classes(
                btn, btn_styles.outline, btn_sizes.sm,
                flex_display, items.center, gap(1), m.t(4)
            ),
            hx_get=urls.management_page,
            hx_target=f"#{ManagementHtmlIds.PAGE_CONTENT}",
            hx_swap="outerHTML",
        ))
    
    return Div(
        *children,
        id=ManagementHtmlIds.DOCUMENT_DETAIL,
        cls=combine_classes(
            flex_display, flex_direction.col, items.center, justify.center,
            p(12)
        )
    )

In [ ]:
err = render_detail_error("Document not found.", urls)
xml = to_xml(err)
assert "Document not found." in xml
assert "Back to List" in xml
print("Error state OK")

Error state OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()